##TrendPulse - Part 2: Numpy & Pandas Data Wrangling



In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
COUNTRIES = ["US", "GB", "IN"]

W_VIEWS = 0.50
W_LIKES = 0.30
W_COMMENTS = 0.20

print(f"TrendPulse Part 2 | countries={COUNTRIES} | started={datetime.now()}")
print(f"Weights: views={W_VIEWS}, likes={W_LIKES}, comments={W_COMMENTS}")

TrendPulse Part 2 | countries=['US', 'GB', 'IN'] | started=2026-11-08 05:55:22.229997
Weights: views=0.5, likes=0.3, comments=0.2


##NumPy arrays



In [ ]:
frames_raw = []
for code in COUNTRIES:
    path = DATA_DIR / f"{code}videos.csv"
    tmp = pd.read_csv(path)
    tmp["country"] = code
    frames_raw.append(tmp)
    print(f"Loaded {code}: {tmp.shape}")

raw = pd.concat(frames_raw, ignore_index=True)
print(f"Combined raw shape: {raw.shape}")

views = raw["views"].to_numpy(dtype=float)
likes = raw["likes"].to_numpy(dtype=float)
dislikes = raw["dislikes"].to_numpy(dtype=float)
comments = raw["comment_count"].to_numpy(dtype=float)

for name, arr in [("views", views), ("likes", likes), ("dislikes", dislikes), ("comments", comments)]:
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")

print("\n10 largest view counts:", np.sort(views[~np.isnan(views)])[-10:][::-1])
print("Every 100th view (first 5 shown):", views[::100][:5])
print("Custom sub-array views[100:110]:", views[100:110])

Loaded US: (16080, 17)
Loaded GB: (15075, 17)
Loaded IN: (15075, 17)
Combined raw shape: (46230, 17)
views: shape=(46230,), dtype=float64
likes: shape=(46230,), dtype=float64
dislikes: shape=(46230,), dtype=float64
comments: shape=(46230,), dtype=float64

10 largest view counts: [3.60760164e+08 3.43297002e+08 2.86714357e+08 2.82812154e+08
 2.55362869e+08 2.36275528e+08 2.30704137e+08 2.10747294e+08
 2.00444300e+08 1.81785943e+08]
Every 100th view (first 5 shown): [ 48555.  21869. 245258. 421352.  54521.]
Custom sub-array views[100:110]: [ 21869.  97158. 168536. 183654.  87527. 216944. 185473.  63499.  32895.
 135582.]


##Engagement score without loops

In [ ]:
views_clean = np.nan_to_num(views, nan=0.0)
likes_clean = np.nan_to_num(likes, nan=0.0)
dislikes_clean = np.nan_to_num(dislikes, nan=0.0)
comments_clean = np.nan_to_num(comments, nan=0.0)

like_denom = likes_clean + dislikes_clean
like_ratio = np.where(like_denom > 0, likes_clean / like_denom, 0.0)

comment_rate = np.where(views_clean > 0, comments_clean / views_clean, 0.0)

print("like_ratio first 5:", like_ratio[:5])
print("comment_rate first 5:", comment_rate[:5])


def min_max_normalize(values):
    """Scale numbers to the range 0 to 1."""
    smallest = np.min(values)
    largest = np.max(values)
    spread = largest - smallest
    if spread == 0:
        return np.zeros_like(values)
    return (values - smallest) / spread


views_norm = min_max_normalize(views_clean)
likes_norm = min_max_normalize(likes_clean)
comments_norm = min_max_normalize(comments_clean)

engagement = (W_VIEWS * views_norm) + (W_LIKES * likes_norm) + (W_COMMENTS * comments_norm)
print("engagement first 5:", engagement[:5])
print("engagement min:", engagement.min(), "max:", engagement.max())

log_views = np.log1p(views_clean)
print("log_views first 5:", log_views[:5])

cutoff = np.quantile(engagement, 0.90)
top10_mask = engagement >= cutoff
print("Top 10% cutoff:", cutoff)
print("How many in top 10%:", top10_mask.sum())

like_ratio first 5: [0.92401857 0.70199879 0.90703465 0.95652174 0.98544282]
comment_rate first 5: [0.00280095 0.00728886 0.00186483 0.00235061 0.00238771]
engagement first 5: [1.24609284e-04 6.57639901e-04 5.25824632e-04 1.65245610e-05
 8.67803150e-04]
engagement min: 5.619756251089524e-07 max: 0.953804648055602
log_views first 5: [10.79047305 12.19796991 12.35301311  8.99764208 12.82976792]
Top 10% cutoff: 0.005395327059212906
How many in top 10%: 4623


/tmp/ipykernel_480/3958199646.py:7: RuntimeWarning: invalid value encountered in divide
  like_ratio = np.where(like_denom > 0, likes_clean / like_denom, 0.0)


##Reshape and broadcast


In [ ]:
daily_counts = raw.groupby(["trending_date", "country"]).size()
daily_table = daily_counts.unstack(fill_value=0)

daily_table = daily_table.reindex(columns=COUNTRIES, fill_value=0)

day_country = daily_table.to_numpy(dtype=float)
print("day_country shape:", day_country.shape)
print("First 3 days:\n", day_country[:3])

flat = day_country.flatten()
print("flatten length:", len(flat))

n_days = day_country.shape[0]
n_countries = day_country.shape[1]
back = flat.reshape(n_days, n_countries)
print("reshape matches original?", np.array_equal(back, day_country))

day_total = day_country.sum(axis=1, keepdims=True)
country_share = np.where(day_total > 0, day_country / day_total, 0.0)
print("Share on first day (US, GB, IN):", country_share[0])

country_avg = day_country.mean(axis=0, keepdims=True)
vs_avg = np.where(country_avg > 0, day_country / country_avg, 0.0)
print("Vs country average on first day:", vs_avg[0])

day_country shape: (80, 3)
First 3 days:
 [[201. 201. 200.]
 [200. 200. 201.]
 [200. 200. 202.]]
flatten length: 240
reshape matches original? True
Share on first day (US, GB, IN): [0.33388704 0.33388704 0.33222591]
Vs country average on first day: [1.         1.06666667 1.06135987]


##Loading and Inspection


In [ ]:
country_frames = []

for code in COUNTRIES:
    file_path = DATA_DIR/(code + "videos.csv")
    df = pd.read_csv(file_path)
    df["country"] = code
    country_frames.append(df)

i = 0
while i < len(country_frames):
    df = country_frames[i]
    code = COUNTRIES[i]
    print("=" * 50)
    print("Country:", code)
    print("shape:", df.shape)
    print(df.head())
    print(df.info())
    print(df.describe())
    i = i + 1

combined = pd.concat(country_frames, ignore_index=True)
print("=" * 50)
print("COMBINED shape:", combined.shape)
print(combined.head())
print(combined.info())
print(combined.describe())

Country: US
shape: (16080, 17)
      video_id trending_date                                  title  \
0  g_N-w8LeAi8      17.14.11           Honest Football Prank LEAKED   
1  Ugyql2NlPAd      17.14.11              Surprising Cricket Review   
2  8fE19jkxJ9n      17.14.11  Mira Solis - Football (Live) OFFICIAL   
3  WOY0nn7HCsO      17.14.11     Zed Rivera - Space (Lyric Video)?!   
4  2Lc_Cs5VVr_      17.14.11             Surprising Election Recipe   

           channel_title  category_id              publish_time  \
0          QuietSpeech10           22  2017-11-14T07:04:28.000Z   
1  IncredibleChallenge41           25  2017-11-12T22:35:48.000Z   
2         WildTutorial42           10  2017-11-13T08:40:22.000Z   
3    ShockingInterview30           10  2017-10-27T22:21:47.000Z   
4         UltimateVlog96           27  2017-11-09T13:20:20.000Z   

              tags   views    likes  dislikes  comment_count  \
0   Monsoon|Robots   48555   2189.0       180          136.0   
1          

In [ ]:
combined["trending_date"] = pd.to_datetime(
    combined["trending_date"],
    format="%y.%d.%m",
    errors="coerce",
)

combined["publish_time"] = pd.to_datetime(
    combined["publish_time"],
    errors="coerce",
    utc=True,
)

number_cols = ["views", "likes", "dislikes", "comment_count", "category_id"]
for col in number_cols:
    combined[col] = pd.to_numeric(combined[col], errors="coerce")

print(combined.dtypes)
print("trending_date examples:", combined["trending_date"].head(3).tolist())
print("publish_time examples:", combined["publish_time"].head(3).tolist())

video_id                               object
trending_date                  datetime64[ns]
title                                  object
channel_title                          object
category_id                             int64
publish_time              datetime64[ns, UTC]
tags                                   object
views                                   int64
likes                                 float64
dislikes                                int64
comment_count                         float64
thumbnail_link                         object
comments_disabled                        bool
ratings_disabled                         bool
video_error_or_removed                   bool
description                            object
country                                object
dtype: object
trending_date examples: [Timestamp('2017-11-14 00:00:00'), Timestamp('2017-11-14 00:00:00'), Timestamp('2017-11-14 00:00:00')]
publish_time examples: [Timestamp('2017-11-14 07:04:28+0000', tz='UTC'), Time

##Data-quality issues (and how we handle them)

**Missing likes / comment_count** → fillna(0)

**Missing description** → fillna("")

**Bad dates after parsing** → dropna(subset=["trending_date", "publish_time"])

**Duplicate rows** (same video_id + trending_date + country) → drop_duplicates, keep first

**Odd date format** for trending_date → parse with %y.%d.%m

**publish_time as text** → convert with pd.to_datetime

**Removed / error videos** → keep the rows (flag is still useful)

##Filtering

In [ ]:
v = combined["views"].fillna(0).to_numpy(dtype=float)
l = combined["likes"].fillna(0).to_numpy(dtype=float)
cmt = combined["comment_count"].fillna(0).to_numpy(dtype=float)

combined["engagement_score"] = (
    W_VIEWS * min_max_normalize(v)
    + W_LIKES * min_max_normalize(l)
    + W_COMMENTS * min_max_normalize(cmt)
)

mask_a = (combined["views"] >= 1000000) & (combined["likes"] > 50000)
answer_a = combined[mask_a]
print("(a) rows with 1M+ views and 50k+ likes:", len(answer_a))
print(answer_a.loc[:, ["title", "country", "views", "likes"]].head())

answer_b = combined.sort_values("engagement_score", ascending=False).head(20)
print("\n(b) Top 20 by engagement_score:")
print(answer_b.loc[:, ["title", "channel_title", "country", "views", "engagement_score"]])

median_views = combined["views"].median()
mask_c = (
    ((combined["category_id"] == 10) | (combined["category_id"] == 20))
    & (combined["views"] > median_views)
)
answer_c = combined[mask_c]
print("\n(c) Music/Gaming above median views", median_views, ":", len(answer_c), "rows")
print(answer_c.loc[:, ["title", "category_id", "views", "country"]].head())

first_index = combined.index[0]
print("\nloc title:", combined.loc[first_index, "title"])
print("iloc first 3 rows, first 4 columns:")
print(combined.iloc[0:3, 0:4])


(a) rows with 1M+ views and 50k+ likes: 6757
                                 title country    views     likes
7       Official Cats Speech EXCLUSIVE      US  2841278   52635.0
11  Surprising Budget Interview WATCH!      US  2855690  160954.0
13          DJ Aster - Festival (Live)      US  1905264   75034.0
19           Shocking Monsoon Reaction      US  1130682   78147.0
55             Epic Election Interview      US  2998582  120472.0

(b) Top 20 by engagement_score:
                                                  title  \
26526                       Lena Ortiz - Monsoon (Live)   
26325                       Lena Ortiz - Monsoon (Live)   
30954  The Hollow Pines - Phones (Official Video) WATCH   
24108       Rook & Vale - Phones (Official Video) WATCH   
26123                       Lena Ortiz - Monsoon (Live)   
30753  The Hollow Pines - Phones (Official Video) WATCH   
25922                       Lena Ortiz - Monsoon (Live)   
23907       Rook & Vale - Phones (Official Video) WATC

##Missing values and duplicates


In [ ]:
n_rows = len(combined)
print("Total rows before missing/duplicate cleaning:", n_rows)

missing_count = combined.isna().sum()

print("\nMissing values (count and percent):")
for col in combined.columns:
    count = int(missing_count[col])
    if count > 0:
        percent = (count / n_rows) * 100
        print(col, ":", count, "(", round(percent, 2), "%)")

combined["likes"] = combined["likes"].fillna(0)
combined["comment_count"] = combined["comment_count"].fillna(0)
combined["description"] = combined["description"].fillna("")

print("\nWhy fill likes/comments with 0?")
print("So we can still score the row instead of deleting useful video info.")
print("Why fill description with empty text?")
print("We do not need description for the numeric questions in this Part.")

rows_before_dropna = len(combined)
combined = combined.dropna(subset=["trending_date", "publish_time"])
rows_after_dropna = len(combined)
print("\nRows dropped by dropna (bad dates):", rows_before_dropna - rows_after_dropna)
print("Why dropna on dates? Without dates we cannot compute days_to_trend.")

dup_flags = combined.duplicated(
    subset=["video_id", "trending_date", "country"],
    keep="first",
)
n_dups = int(dup_flags.sum())
print("\nDuplicate rows found:", n_dups)

combined = combined.drop_duplicates(
    subset=["video_id", "trending_date", "country"],
    keep="first",
)
print("Rows after removing duplicates:", len(combined))
print("Duplicate means: same video_id + trending_date + country.")

v = combined["views"].to_numpy(dtype=float)
l = combined["likes"].to_numpy(dtype=float)
cmt = combined["comment_count"].to_numpy(dtype=float)
combined["engagement_score"] = (
    W_VIEWS * min_max_normalize(v)
    + W_LIKES * min_max_normalize(l)
    + W_COMMENTS * min_max_normalize(cmt)
)
print("\nEngagement score refreshed after cleaning.")


Total rows before missing/duplicate cleaning: 46230

Missing values (count and percent):
likes : 12 ( 0.03 %)
comment_count : 36 ( 0.08 %)
description : 1624 ( 3.51 %)

Why fill likes/comments with 0?
So we can still score the row instead of deleting useful video info.
Why fill description with empty text?
We do not need description for the numeric questions in this Part.

Rows dropped by dropna (bad dates): 0
Why dropna on dates? Without dates we cannot compute days_to_trend.

Duplicate rows found: 230
Rows after removing duplicates: 46000
Duplicate means: same video_id + trending_date + country.

Engagement score refreshed after cleaning.


##Joining Category names

In [ ]:
categories = pd.read_csv(DATA_DIR / "categories.csv")
print(categories.head())

rows_before = len(combined)
combined = combined.merge(categories, on="category_id", how="left")
rows_after = len(combined)

print("Rows before merge:", rows_before)
print("Rows after merge:", rows_after)
print("Rows lost:", rows_before - rows_after)
print(combined.loc[:, ["category_id", "category_name"]].head())

   category_id     category_name
0            1  Film & Animation
1            2  Autos & Vehicles
2           10             Music
3           15    Pets & Animals
4           17            Sports
Rows before merge: 46000
Rows after merge: 46000
Rows lost: 0
   category_id    category_name
0           22   People & Blogs
1           25  News & Politics
2           10            Music
3           10            Music
4           27        Education


##Aggregation

In [ ]:
by_category = combined.groupby("category_name").agg(
    record_count=("video_id", "size"),
    avg_views=("views", "mean"),
    avg_likes=("likes", "mean"),
    avg_engagement=("engagement_score", "mean"),
)
by_category = by_category.sort_values("record_count", ascending=False)
print("By category:")
print(by_category)

by_country = combined.groupby("country").agg(
    record_count=("video_id", "size"),
    avg_views=("views", "mean"),
    avg_likes=("likes", "mean"),
    avg_engagement=("engagement_score", "mean"),
)
print("\nBy country:")
print(by_country)

counts = combined["category_name"].value_counts()
percents = combined["category_name"].value_counts(normalize=True) * 100
print("\nCategory counts:")
print(counts)
print("\nCategory percents:")
print(percents.round(2))

combined["publish_weekday"] = combined["publish_time"].dt.day_name()
combined["publish_hour"] = combined["publish_time"].dt.hour

weekday_views = combined.groupby("publish_weekday")["views"].mean()
weekday_views = weekday_views.sort_values(ascending=False)
print("\nAverage views by publish weekday:")
print(weekday_views)

hour_views = combined.groupby("publish_hour")["views"].mean()
hour_views = hour_views.sort_values(ascending=False)
print("\nTop publish hours by average views:")
print(hour_views.head())

By category:
                      record_count     avg_views      avg_likes  \
category_name                                                     
Entertainment                12270  9.010256e+05   28932.229992   
Music                        10310  3.190950e+06  134940.252182   
People & Blogs                3804  5.512341e+05   19835.408780   
Comedy                        3258  6.034757e+05   28692.014733   
News & Politics               2422  2.837947e+05    3110.593724   
Film & Animation              2297  5.779740e+05   17191.443622   
Gaming                        2057  5.096649e+05   20747.268838   
Sports                        1921  3.749782e+05   10451.312858   
Howto & Style                 1825  2.996848e+05   10216.850959   
Education                     1645  3.627163e+05   12615.622492   
Science & Technology          1562  3.563504e+05   11354.096031   
Autos & Vehicles              1007  2.796927e+05    8705.551142   
Pets & Animals                 913  3.497486e+05 

##Time to trend and trending lifespan


In [ ]:
publish_day = combined["publish_time"].dt.tz_convert(None).dt.normalize()
trend_day = combined["trending_date"]

combined["days_to_trend"] = (trend_day - publish_day).dt.days

print("days_to_trend distribution:")
print(combined["days_to_trend"].describe())

lifespan = combined.groupby(["country", "video_id", "title"])["trending_date"].nunique()
lifespan = lifespan.reset_index()
lifespan = lifespan.rename(columns={"trending_date": "trending_lifespan_days"})

longest = lifespan.sort_values("trending_lifespan_days", ascending=False).head(10)
print("\n10 longest-lasting videos:")
print(longest)

video_category = combined.groupby(["country", "video_id"], as_index=False).agg(
    category_name=("category_name", "first")
)
lifespan_with_cat = lifespan.merge(video_category, on=["country", "video_id"], how="left")

avg_life = lifespan_with_cat.groupby("category_name")["trending_lifespan_days"].mean()
avg_life = avg_life.sort_values(ascending=False)
print("\nAverage lifespan by category:")
print(avg_life)

days_to_trend distribution:
count    46000.000000
mean         5.090457
std          6.243901
min          0.000000
25%          1.000000
50%          3.000000
75%          6.000000
max         47.000000
Name: days_to_trend, dtype: float64

10 longest-lasting videos:
      country     video_id                                         title  \
1611       GB  NqokCmZykzK                Anya Lux - Paris (Lyric Video)   
1630       GB  O6zdCRwWdf1            MIRA SOLIS - WINTER (LYRIC VIDEO)?   
1019       GB  EZaG9_2Yi1u                       KAIRO - FESTIVAL (LIVE)   
950        GB  DYLzrqG-SrK                      Official Football Update   
1846       GB  RdqSDDng84n                   Lena Ortiz - Monsoon (Live)   
1432       GB  Kv1WBQp86cP                 Sunny Park - Cats (Live) MUST   
3859       GB  w1lZB-FX3Sn  DJ Aster - Festival (Official Video) WATCH?!   
11006      US  BsTMdCTYs7P                      Surprising Guitar Update   
2343       GB  Z1aFIGyR5zv           Nova Kane -

##Combining Data


In [ ]:
us_df = combined[combined["country"] == "US"].copy()
gb_df = combined[combined["country"] == "GB"].copy()
in_df = combined[combined["country"] == "IN"].copy()

stacked = pd.concat([us_df, gb_df, in_df], ignore_index=True)
print("Stacked shape after concat:", stacked.shape)

per_video = stacked.groupby(["country", "video_id", "channel_title"], as_index=False).agg(
    video_views=("views", "max"),
    days_on_list=("trending_date", "nunique"),
)

per_channel = stacked.groupby(["country", "channel_title"], as_index=False).agg(
    n_trending_videos=("video_id", "nunique"),
    total_views=("views", "sum"),
)

channel_video = per_video.merge(
    per_channel,
    on=["country", "channel_title"],
    how="left",
)

print("Sample of merged video + channel summary:")
cols_to_show = [
    "country",
    "channel_title",
    "video_id",
    "n_trending_videos",
    "total_views",
    "video_views",
]
print(channel_video.loc[:, cols_to_show].head(10))

print("\nPart 2 finished successfully.")

Stacked shape after concat: (46000, 22)
Sample of merged video + channel summary:
  country          channel_title     video_id  n_trending_videos  total_views  \
0      GB     UltimateUnboxing65  --UQXlrk5E0                138    445143715   
1      GB       QuietChallenge75  --pb5oKkYrH                208   2706289596   
2      GB  SurprisingChallenge82  -13iTcoyyzU                 41     13977297   
3      GB     UltimateUnboxing65  -1VjLKhzB3I                138    445143715   
4      GB       FinalBreakdown11  -1aX3Wk1cJx                 38    125493995   
5      GB   UltimateHighlights98  -1rsv0uY-8K                  2     19206300   
6      GB        PerfectUpdate53  -3WfJZgVB5h                 23    472690844   
7      GB         HonestSpeech62  -48JdRO2P5_                 19      8395325   
8      GB         FinalTrailer97  -5gYZ79gyl-                 12      4444629   
9      GB        PerfectRecipe71  -5iZ5HWgnCs                 41     28641274   

   video_views  
0       1